# Yeouido Weather by Year

- Location: Yeouido (`lat=37.521624`, `lng=126.924191`)
- Source: Open-Meteo archive API
- Note: `now` from the request is interpreted as `snow`.
- Any days that are not rainy, snow, clear, or foggy are labeled as `cloudy` so every date has a usable weather value.

In [1]:
from __future__ import annotations

import json
import ssl
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen

import pandas as pd

LATITUDE = 37.521624
LONGITUDE = 126.924191
TIMEZONE = "Asia/Seoul"
API_URL = "https://archive-api.open-meteo.com/v1/archive"
BASE_DIR = Path.cwd()
OUTPUT_FILES = {
    2023: BASE_DIR / "weather_2023.csv",
    2024: BASE_DIR / "weather_2024.csv",
}

CLEAR_CODES = {0, 1}
FOGGY_CODES = {45, 48}
SNOW_CODES = {71, 73, 75, 77, 85, 86}
RAINY_CODES = {51, 53, 55, 56, 57, 61, 63, 65, 66, 67, 80, 81, 82, 95, 96, 99}


In [2]:
def classify_weather(weather_code: int, precipitation_sum: float, snowfall_sum: float) -> str:
    if snowfall_sum > 0 or weather_code in SNOW_CODES:
        return "snow"
    if weather_code in FOGGY_CODES:
        return "foggy"
    if precipitation_sum > 0 or weather_code in RAINY_CODES:
        return "rainy"
    if weather_code in CLEAR_CODES:
        return "clear"
    return "cloudy"


def fetch_year_weather(year: int) -> pd.DataFrame:
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": f"{year}-01-01",
        "end_date": f"{year}-12-31",
        "daily": "weather_code,temperature_2m_mean,precipitation_sum,snowfall_sum",
        "timezone": TIMEZONE,
    }
    url = f"{API_URL}?{urlencode(params)}"

    with urlopen(url, timeout=60, context=ssl.create_default_context()) as response:
        payload = json.load(response)

    daily = payload["daily"]
    frame = pd.DataFrame(
        {
            "date": pd.to_datetime(daily["time"]).strftime("%Y-%m-%d"),
            "weather_code": pd.to_numeric(daily["weather_code"]),
            "temperature_c": pd.to_numeric(daily["temperature_2m_mean"]),
            "precipitation_sum": pd.to_numeric(daily["precipitation_sum"]),
            "snowfall_sum": pd.to_numeric(daily["snowfall_sum"]),
        }
    )
    frame["weather"] = frame.apply(
        lambda row: classify_weather(
            int(row["weather_code"]),
            float(row["precipitation_sum"]),
            float(row["snowfall_sum"]),
        ),
        axis=1,
    )
    return frame[["date", "weather", "temperature_c", "weather_code", "precipitation_sum", "snowfall_sum"]]


In [3]:
yearly_frames = {}

for year, output_path in OUTPUT_FILES.items():
    frame = fetch_year_weather(year)
    frame.to_csv(output_path, index=False, encoding="utf-8-sig")
    yearly_frames[year] = frame
    print(f"Saved {output_path.name}: {len(frame)} rows")
    print(frame.head())
    print(frame["weather"].value_counts(dropna=False).sort_index())
    print()


Saved weather_2023.csv: 365 rows
         date weather  temperature_c  weather_code  precipitation_sum  \
0  2023-01-01  cloudy           -2.6             3                0.0   
1  2023-01-02  cloudy           -6.1             3                0.0   
2  2023-01-03   clear           -6.5             0                0.0   
3  2023-01-04  cloudy           -4.3             3                0.0   
4  2023-01-05  cloudy           -3.4             3                0.0   

   snowfall_sum  
0           0.0  
1           0.0  
2           0.0  
3           0.0  
4           0.0  
weather
clear      31
cloudy    165
rainy     154
snow       15
Name: count, dtype: int64



Saved weather_2024.csv: 366 rows
         date weather  temperature_c  weather_code  precipitation_sum  \
0  2024-01-01  cloudy            0.8             3                0.0   
1  2024-01-02   rainy            1.2            51                0.2   
2  2024-01-03    snow            0.1            71                0.6   
3  2024-01-04  cloudy           -0.3             3                0.0   
4  2024-01-05   rainy            2.8            51                0.5   

   snowfall_sum  
0          0.00  
1          0.00  
2          0.07  
3          0.00  
4          0.00  
weather
clear      29
cloudy    150
rainy     161
snow       26
Name: count, dtype: int64

